# Dependence on the Ising coupling

$\chi$ at $E_g + 0.1$ eV (TNN, GGA, all three field terms) for $\lambda$ from $\lambda_0/2$ to $3\lambda_0/2$, everything else fixed. The $\lambda$ values are those of the production sweep: `n_lam` points across $[\lambda_0/2, 3\lambda_0/2]$ merged with `n_lam` points across $[0, 0.4]$ eV, keeping the ones inside the window. $E_g$ is recomputed for each $\lambda$. Since $\eta$ is odd in $B$, $\chi = \eta(10\,{\rm T})/10$ T.

## Load engine

In [ ]:
%run engine.ipynb

## Settings

In [ ]:
R_disk = 0.9       # disk radius around each valley (1/a)
h_lam = 0.002      # disk spacing (0.005 for a quick look; 0.01 gives wrong signs)
n_lam = 21         # points in each of the two lambda ranges (5 for a quick look)

valley_disk = disk(R_disk, h_lam)

## $\chi(\lambda)$

On the valley disks with $\nu = 5$ meV. About 75 s per value of $\lambda$ (25 to 33 values per material), 10 h in all.

In [ ]:
for m in materials:
    file = f"data/lambda_{m}_R{R_disk}_h{h_lam}_n{n_lam}.npz"
    if os.path.exists(file):
        continue
    lam0 = lambda_SOC[m]
    lams = np.unique(np.round(np.concatenate([np.linspace(0.5 * lam0, 1.5 * lam0, n_lam), np.linspace(0, 0.4, n_lam)]), 6))
    lams = lams[(lams >= 0.5 * lam0 - 1e-9) & (lams <= 1.5 * lam0 + 1e-9)]
    chi = []
    for lam in lams:
        eta = eta_mag(m, "TNN", 10.0, np.array([gap(m, lam=lam) + 0.1]), 0.005, valley_disk, centers=[K_len, -K_len], lam=lam)
        chi.append(eta[0, 0, 0, 0] / 10)
    np.savez(file, lam=lams, chi=eta_SI(params(m)["a"]) * np.array(chi))

## Figure

Points: the calculation. Solid and dashed: straight-line and quadratic fits. Dotted: the physical $\lambda_0$.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for j, m in enumerate(materials):
    d = np.load(f"data/lambda_{m}_R{R_disk}_h{h_lam}_n{n_lam}.npz")
    lam, chi = d["lam"], 1e4 * d["chi"]
    print(f"{m}: chi(lambda_0) = {np.interp(lambda_SOC[m], lam, chi):+.3f} x 1e-4, straight-line slope {np.polyfit(lam, chi, 1)[0]:+.2f} x 1e-4 per eV")
    axes[j].plot(lam, chi, "o")
    axes[j].plot(lam, np.polyval(np.polyfit(lam, chi, 1), lam), "-")
    axes[j].plot(lam, np.polyval(np.polyfit(lam, chi, 2), lam), "--")
    axes[j].axvline(lambda_SOC[m], color="gray", ls=":")
    axes[j].axhline(0, color="gray", lw=0.5)
    axes[j].set_title(m)
    axes[j].set_xlabel(r"$\lambda$ (eV)")
axes[0].set_ylabel(r"$\chi$ ($10^{-4}$ A m V$^{-2}$ s$^{-1}$ T$^{-1}$)")
plt.tight_layout()
plt.savefig("figures/lpge_lambda_grid.png", dpi=200)
plt.show()